# dataloader-pin-memory-workers — worked example 3: Custom collate_fn to pad variable-length items

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-pin-memory-workers`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The default `collate_fn` stacks items by calling `torch.stack`, which requires every item to share a shape. When items have *different* lengths you supply a custom `collate_fn(batch)` that pads each item up to the batch maximum and stacks the result. The function receives a Python list of dataset items and returns the assembled batch tensor(s).

## Worked solution

Goal: a loader whose items are 1-D vectors of *varying* length, collated into one padded `(B, Lmax)` tensor.

1. **Build a ragged dataset.** A tiny custom `Dataset` returns vectors of length 2, 5, 3, 4 — the default collate would crash trying to stack mismatched shapes.
2. **Write `pad_collate`.** It gets a list of 1-D tensors. Compute `Lmax = max(len)`. For each item, right-pad with zeros to `Lmax` using `torch.nn.functional.pad(item, (0, Lmax - len))`, then `torch.stack` the padded list into `(B, Lmax)`.
3. **Why pad inside collate, not in the dataset?** Padding to the *batch* max (not the global max) keeps tensors small and adapts per batch — the standard NLP/sequence pattern.
4. **Wire it in.** Pass `collate_fn=pad_collate`, `num_workers=0` (the closure can't pickle across worker processes in a notebook), `shuffle=False` for a deterministic check.
5. **Verify.** With `batch_size=4` and one batch of the four items, `Lmax=5`, so the batch shape is `(4, 5)`, and the row that was length 2 ends in three zeros.

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

class RaggedDS(Dataset):
    def __init__(self, lengths):
        self.items = [t.arange(1, L + 1, dtype=t.float32) for L in lengths]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def pad_collate(batch):
    Lmax = max(item.shape[0] for item in batch)
    padded = [F.pad(item, (0, Lmax - item.shape[0])) for item in batch]
    return t.stack(padded, dim=0)

def make_ragged_loader(dataset, batch_size):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=pad_collate,
        num_workers=0,
        pin_memory=False,
        shuffle=False,
    )

t.manual_seed(0)
ds = RaggedDS([2, 5, 3, 4])
loader = make_ragged_loader(ds, batch_size=4)
batch = next(iter(loader))
print('batch shape:', tuple(batch.shape), '(expected (4, 5))')
print('first row (was length 2):', batch[0].tolist())